In [ ]:
# Matrix multiplication with numba(GPU) and numba(CPU)
# Import and initialize numpy, numba and time
import numpy as np
import time 
from numba import njit, prange, cuda


In [ ]:
#################### Array size
N = 400

In [ ]:
#################### Create some space on CPU/HOST (random 32-bit ints)
a_cpu = np.random.uniform(1.0, 100.0, size=(N,N)).astype(np.uint32) 
b_cpu = np.random.uniform(1.0, 100.0, size=(N,N)).astype(np.uint32)
c_cpu = np.zeros((N,N), np.uint32)

In [ ]:
@cuda.jit
def multiplication(a_gpu, b_gpu, c_gpu, N):
	col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
	row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
	if row < N and col < N:
		tmp = 0
		for k in range(N):
			tmp += a_gpu[row * N + k] * b_gpu[k * N + col]
		c_gpu[row * N + col] = tmp


In [ ]:
#################### Allocate and transfer data to GPU
a_gpu = cuda.to_device(a_cpu.reshape(-1))
b_gpu = cuda.to_device(b_cpu.reshape(-1))
c_gpu = cuda.device_array(a_cpu.size, dtype=np.uint32)


In [ ]:
#################### Define Numba CUDA kernel
block_size = 32
grid_size = ( (N + block_size - 1) // block_size,
              (N + block_size - 1) // block_size )

In [ ]:
#################### Launch GPU kernel and time it
start_gpu = time.time()
multiplication[grid_size, (block_size, block_size)](a_gpu, b_gpu, c_gpu, N)
cuda.synchronize()
gpu_time = time.time() - start_gpu

#################### Copy back and reshape
c_gpu_res = c_gpu.copy_to_host().reshape(N, N)

print("Elapsed time using GPU Numba (sec): ", gpu_time)
print("---------------------")